In [6]:
"""
The MIT License (MIT)
Copyright (c) 2021 NVIDIA
Permission is hereby granted, free of charge, to any person obtaining a copy of
this software and associated documentation files (the "Software"), to deal in
the Software without restriction, including without limitation the rights to
use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of
the Software, and to permit persons to whom the Software is furnished to do so,
subject to the following conditions:
The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.
THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS
FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR
COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER
IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN
CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
"""


'\nThe MIT License (MIT)\nCopyright (c) 2021 NVIDIA\nPermission is hereby granted, free of charge, to any person obtaining a copy of\nthis software and associated documentation files (the "Software"), to deal in\nthe Software without restriction, including without limitation the rights to\nuse, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of\nthe Software, and to permit persons to whom the Software is furnished to do so,\nsubject to the following conditions:\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS\nFOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR\nCOPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER\nIN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OU

This code example is very similar to c5e1_mnist_learning but the network is modified to use ReLU neruons in the hidden layer, softmax in the output layer, categorical crossentropy as loss function, Adam as optimizer, and a mini-batch size of 64. More context for this code example can be found in the section "Experiment: Tweaking Network and Learning Parameters" in Chapter 5 in the book Learning Deep Learning by Magnus Ekman (ISBN: 9780137470358).

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
import numpy as np

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

EPOCHS     = 10
BATCH_SIZE = 64
LR         = 1e-3
INPUT_DIM  = 784
HIDDEN_DIM = 25
OUTPUT_DIM = 10

# Load training dataset into a single batch to compute mean and stddev.
transform = transforms.Compose([transforms.ToTensor()])
trainset = MNIST(root='./pt_data', train=True, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=len(trainset), shuffle=True)
data = next(iter(trainloader))
mean = data[0].mean().item()
stddev = data[0].std().item()

# Helper function needed to standardize data when loading datasets.
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize(mean, stddev)])

trainset = MNIST(root='./pt_data', train=True, download=True, transform=transform)
testset = MNIST(root='./pt_data', train=False, download=True, transform=transform)


Using device: cpu


In this example we want to use ReLU as activation for the first layer and softmax for the second layer. One thing that is slightly unintuitive is that we omit the activation function altogether for the output layer. The reason is that the cross-entropy loss function implementation in PyTorch is implemented using the inputs to the softmax (also known as logits) instead of the outputs. This makes for an implementation that is more numerically stable (also see some more details in the section "Computer implementation of the cross-entropy loss function" in Chapter 5 in the book).

Apart from changing the layer types, we also change the weight initialization scheme to Kaiming (He) for the ReLU layer and Xavier (Glorot) for the softmax layer.


In [8]:
# Define model. Final activation is omitted since softmax is part of
# cross-entropy loss function in PyTorch.
class FeedForwardNet(nn.Module):
    def __init__(
        self,
        input_dim:  int = INPUT_DIM,
        hidden_dim: int = HIDDEN_DIM,
        output_dim: int = OUTPUT_DIM
    ) -> None:
        super().__init__()
        self.hidden = nn.Linear(input_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, output_dim)
        self._initialize_weights()

    def _initialize_weights(self) -> None:
        # Kaiming (He) initialization for the hidden layer.
        nn.init.kaiming_normal_(self.hidden.weight)
        nn.init.constant_(self.hidden.bias, 0.0)
        # Xavier (Glorot) initialization for the output layer (no activation).
        nn.init.xavier_uniform_(self.output.weight)
        nn.init.constant_(self.output.bias, 0.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.hidden(x))
        return self.output(x)    # Raw logits; softmax applied by loss function.

model = FeedForwardNet()


The training loop looks very similar to c5e1_mnist_learning. We change to using the Adam optimizer and the CrossEntropyLoss function.

Additionally, inside the training loop we no longer convert the targets to one-hot encoded targets. The reason is that the CrossEntropyLoss function object is implemented in a way where it assumes that the input is an integer specifying the index to be hot instead of assuming a one-hot encoded input.


In [ ]:
# Use the Adam optimizer.
# Cross-entropy loss as loss function.
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_function = nn.CrossEntropyLoss()

# Transfer model to GPU.
model.to(device)

# Create DataLoader objects that will help create mini-batches.
trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)
testloader  = DataLoader(testset,  batch_size=BATCH_SIZE, shuffle=False)

def run_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer | None,
    device:    torch.device,
    training:  bool
) -> tuple[float, float]:
    """Run one full pass over the DataLoader.
    Pass optimizer=None during evaluation.
    Returns (mean loss, accuracy).
    """
    model.train() if training else model.eval()

    total_loss    = 0.0
    total_correct = 0
    total_samples = 0

    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)
            # Cross-entropy loss does not need one-hot targets in PyTorch.
            loss = criterion(outputs, targets)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # Accumulate metrics.
            _, predicted   = torch.max(outputs, dim=1)
            total_correct += (predicted == targets).sum().item()
            total_samples += targets.size(0)
            total_loss    += loss.item()

    return total_loss / len(loader), total_correct / total_samples

# Train the model. In PyTorch we have to implement the training loop ourselves.
for i in range(EPOCHS):
    train_loss, train_acc = run_epoch(
        model, trainloader, loss_function, optimizer, device, training=True
    )
    test_loss, test_acc = run_epoch(
        model, testloader, loss_function, optimizer=None, device=device, training=False
    )
    print(f'Epoch {i+1}/{EPOCHS} loss: {train_loss:.4f} - acc: {train_acc:.4f} - val_loss: {test_loss:.4f} - val_acc: {test_acc:.4f}')


Epoch 1/10 loss: 0.3651 - acc: 0.8907 - val_loss: 0.2167 - val_acc: 0.9355
Epoch 2/10 loss: 0.1934 - acc: 0.9434 - val_loss: 0.1730 - val_acc: 0.9477
